# 18 — Map-Matching Diagnostics & Visualizations

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Roadmap Section 27 & 30:**
> - Visualize local road graph subgraphs
> - Visualize candidate road segment rankings and GNN confidence distributions
> - Inspect topological path continuity under heavy inertial drift

In [ ]:
import os, sys, json, pickle
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.map_matching.road_graph import RoadNetworkGraph

graph_path = PROJECT_ROOT / 'data' / 'OSM' / 'road_graph_coventry.pkl'
plots_dir = PROJECT_ROOT / 'plots' / 'map_matching'
plots_dir.mkdir(parents=True, exist_ok=True)

if graph_path.exists():
    with open(graph_path, 'rb') as f:
        road_graph = pickle.load(f)
else:
    road_graph = RoadNetworkGraph()
    road_graph.add_node(0, [0, 0])
    road_graph.add_node(1, [100, 0])
    road_graph.add_edge(0, 0, 1)
    road_graph.build_spatial_index()

# Sample query visualization
q_pos = np.array([120.0, 45.0])
cands = road_graph.query_candidate_segments(q_pos, search_radius=100.0, max_candidates=5)

plt.figure(figsize=(9, 7))
for c in cands:
    seg = road_graph.edges[c['edge_id']]
    plt.plot([seg.start_coord[0], seg.end_coord[0]], [seg.start_coord[1], seg.end_coord[1]], 'b-', lw=2.5, label=f'Candidate Edge {c["edge_id"]}')
    plt.plot(c['proj_pos'][0], c['proj_pos'][1], 'mo', markersize=7)
    plt.plot([q_pos[0], c['proj_pos'][0]], [q_pos[1], c['proj_pos'][1]], 'k:', alpha=0.6)

plt.plot(q_pos[0], q_pos[1], 'r*', markersize=14, label='Drifted Query Position')
plt.title('Local Candidate Road Segments & Orthogonal Projections')
plt.xlabel('East (meters)')
plt.ylabel('North (meters)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')

cand_fig = plots_dir / 'gnn_candidate_ranking_S1.png'
plt.savefig(cand_fig, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved candidate ranking diagnostic: {cand_fig}')